# Load Listed Diffusion Models

This notebook loads the pretrained model artifacts listed in `papers_models.md` so analysis code can start from initialized pipelines. It does not generate samples or compute metrics yet.

Models covered here:

- DDPM: `google/ddpm-cifar10-32`
- Stable Diffusion: `stable-diffusion-v1-5/stable-diffusion-v1-5`
- DiT: `facebook/DiT-XL-2-256`
- SiT: configurable local/repo placeholder, because `papers_models.md` lists the SiT paper and GitHub repo but not a specific pretrained checkpoint

## Environment

Run the install cell only if the imports fail. Stable Diffusion may also require Hugging Face authentication and accepting the model license on the model page.

In [ ]:
# Uncomment and run if needed.
# !pip install -q "torch" "diffusers[torch]" "transformers" "accelerate" "safetensors" "huggingface_hub" "ipywidgets"

In [1]:
import gc
import os
from pathlib import Path

import torch
from huggingface_hub import whoami

PROJECT_ROOT = Path.cwd()
HF_CACHE_DIR = PROJECT_ROOT / ".hf_cache"
HF_CACHE_DIR.mkdir(exist_ok=True)

MODEL_IDS = {
    "ddpm_cifar10": "google/ddpm-cifar10-32",
    "stable_diffusion_v1_5": "stable-diffusion-v1-5/stable-diffusion-v1-5",
    "dit_xl_2_256": "facebook/DiT-XL-2-256",
}

def detect_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = detect_device()
torch_dtype = torch.float16 if device.type == "cuda" else torch.float32

print(f"Using device: {device}")
print(f"Using torch dtype: {torch_dtype}")

Using device: mps
Using torch dtype: torch.float32


In [2]:
try:
    user = whoami()
    print(f"Hugging Face authenticated as: {user.get('name', user)}")
except Exception as exc:
    print("Not authenticated with Hugging Face in this environment.")
    print("If Stable Diffusion loading fails, run `huggingface-cli login` in a terminal or `from huggingface_hub import login; login()` here.")
    print(f"Auth check detail: {type(exc).__name__}: {exc}")

Hugging Face authenticated as: sswitz


In [3]:
loaded_models = {}

def move_pipeline_to_device(pipe):
    if device.type == "mps":
        # PyTorch MPS does not benefit from float16 as consistently as CUDA.
        return pipe.to(device)
    return pipe.to(device)

def unload_model(key):
    pipe = loaded_models.pop(key, None)
    if pipe is not None:
        del pipe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def summarize_pipeline(key):
    pipe = loaded_models[key]
    components = sorted(name for name in getattr(pipe, "components", {}).keys())
    print(f"{key}: {pipe.__class__.__name__}")
    print(f"components: {components}")

## DDPM: CIFAR-10 32x32

This is the most lightweight model in the list and is useful for validating the evaluation pipeline before scaling up to larger text/image models.

In [4]:
from diffusers import DDPMPipeline

ddpm_pipe = DDPMPipeline.from_pretrained(
    MODEL_IDS["ddpm_cifar10"],
    cache_dir=HF_CACHE_DIR,
)
ddpm_pipe = move_pipeline_to_device(ddpm_pipe)
loaded_models["ddpm_cifar10"] = ddpm_pipe

summarize_pipeline("ddpm_cifar10")

Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch /Users/samswitz/GitHub/stat542-final-project/.hf_cache/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80: Error no file named diffusion_pytorch_model.safetensors found in directory /Users/samswitz/GitHub/stat542-final-project/.hf_cache/models--google--ddpm-cifar10-32/snapshots/267b167dc01f0e4e61923ea244e8b988f84deb80.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


ddpm_cifar10: DDPMPipeline
components: ['scheduler', 'unet']


## Stable Diffusion v1.5

This is much larger than the CIFAR DDPM. If memory is limited, unload other models before running this cell.

In [5]:
from diffusers import StableDiffusionPipeline

# Optional memory cleanup before loading a large pipeline.
# unload_model("ddpm_cifar10")

sd_pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_IDS["stable_diffusion_v1_5"],
    cache_dir=HF_CACHE_DIR,
    torch_dtype=torch_dtype,
)

if device.type == "cuda":
    sd_pipe.enable_attention_slicing()

sd_pipe = move_pipeline_to_device(sd_pipe)
loaded_models["stable_diffusion_v1_5"] = sd_pipe

summarize_pipeline("stable_diffusion_v1_5")

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

stable_diffusion_v1_5: StableDiffusionPipeline
components: ['feature_extractor', 'image_encoder', 'safety_checker', 'scheduler', 'text_encoder', 'tokenizer', 'unet', 'vae']


## DiT-XL/2 256x256

The listed DiT model is class-conditional. The pipeline loads the model and its scheduler; analysis code can later choose ImageNet class labels for generation.

In [6]:
from diffusers import DiTPipeline

# Optional memory cleanup before loading another large pipeline.
# unload_model("stable_diffusion_v1_5")

dit_pipe = DiTPipeline.from_pretrained(
    MODEL_IDS["dit_xl_2_256"],
    cache_dir=HF_CACHE_DIR,
    torch_dtype=torch_dtype,
)
dit_pipe = move_pipeline_to_device(dit_pipe)
loaded_models["dit_xl_2_256"] = dit_pipe

summarize_pipeline("dit_xl_2_256")

model_index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]

An error occurred while trying to fetch /Users/samswitz/GitHub/stat542-final-project/.hf_cache/models--facebook--DiT-XL-2-256/snapshots/eab87f77abd5aef071a632f08807fbaab0b704d0/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /Users/samswitz/GitHub/stat542-final-project/.hf_cache/models--facebook--DiT-XL-2-256/snapshots/eab87f77abd5aef071a632f08807fbaab0b704d0/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch /Users/samswitz/GitHub/stat542-final-project/.hf_cache/models--facebook--DiT-XL-2-256/snapshots/eab87f77abd5aef071a632f08807fbaab0b704d0/transformer: Error no file named diffusion_pytorch_model.safetensors found in directory /Users/samswitz/GitHub/stat542-final-project/.hf_cache/models--facebook--DiT-XL-2-256/snapshots/eab87f77abd5aef071a632f08807fbaab0b704d0/transformer.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
Expected

dit_xl_2_256: DiTPipeline
components: ['scheduler', 'transformer', 'vae']


## SiT

`papers_models.md` links to the SiT website, paper, and GitHub repository, but does not identify a specific pretrained checkpoint. This cell records that gap and gives a single place to plug in a checkpoint or local clone later.

In [ ]:
SIT_REPO_URL = "https://github.com/willisma/SiT"
SIT_CHECKPOINT = None  # Replace with a local checkpoint path or model artifact once chosen.

if SIT_CHECKPOINT is None:
    print("SiT checkpoint not loaded: no concrete pretrained checkpoint is listed in papers_models.md.")
    print(f"Reference repo: {SIT_REPO_URL}")
else:
    # Add project-specific SiT loading code here after selecting a checkpoint.
    # Example shape:
    #   import sys
    #   sys.path.append("/path/to/SiT")
    #   from models import SiT_XL_2
    #   sit_model = SiT_XL_2(...)
    #   state = torch.load(SIT_CHECKPOINT, map_location="cpu")
    #   sit_model.load_state_dict(state)
    #   sit_model.to(device).eval()
    #   loaded_models["sit"] = sit_model
    raise NotImplementedError("Fill in SiT model construction after choosing a checkpoint.")

## Ready for Analysis

`loaded_models` is the shared registry for downstream analysis cells. It contains whichever model-loading cells have been run successfully in this kernel.

In [7]:
print("Loaded model keys:", sorted(loaded_models.keys()))

analysis_registry = {
    "ddpm_cifar10": {
        "paper_family": "DDPM-style diffusion",
        "model_id": MODEL_IDS["ddpm_cifar10"],
        "pipeline": loaded_models.get("ddpm_cifar10"),
        "output_resolution": 32,
        "conditioning": "unconditional/class dataset prior",
    },
    "stable_diffusion_v1_5": {
        "paper_family": "Stable Diffusion",
        "model_id": MODEL_IDS["stable_diffusion_v1_5"],
        "pipeline": loaded_models.get("stable_diffusion_v1_5"),
        "output_resolution": 512,
        "conditioning": "text prompt",
    },
    "dit_xl_2_256": {
        "paper_family": "DiT",
        "model_id": MODEL_IDS["dit_xl_2_256"],
        "pipeline": loaded_models.get("dit_xl_2_256"),
        "output_resolution": 256,
        "conditioning": "ImageNet class label",
    },
    "sit": {
        "paper_family": "SiT",
        "model_id": None,
        "pipeline": loaded_models.get("sit"),
        "output_resolution": None,
        "conditioning": "checkpoint-dependent",
    },
}

analysis_registry

Loaded model keys: ['ddpm_cifar10', 'dit_xl_2_256', 'stable_diffusion_v1_5']


{'ddpm_cifar10': {'paper_family': 'DDPM-style diffusion',
  'model_id': 'google/ddpm-cifar10-32',
  'pipeline': DDPMPipeline {
    "_class_name": "DDPMPipeline",
    "_diffusers_version": "0.37.1",
    "_name_or_path": "google/ddpm-cifar10-32",
    "scheduler": [
      "diffusers",
      "DDPMScheduler"
    ],
    "unet": [
      "diffusers",
      "UNet2DModel"
    ]
  },
  'output_resolution': 32,
  'conditioning': 'unconditional/class dataset prior'},
 'stable_diffusion_v1_5': {'paper_family': 'Stable Diffusion',
  'model_id': 'stable-diffusion-v1-5/stable-diffusion-v1-5',
  'pipeline': StableDiffusionPipeline {
    "_class_name": "StableDiffusionPipeline",
    "_diffusers_version": "0.37.1",
    "_name_or_path": "stable-diffusion-v1-5/stable-diffusion-v1-5",
    "feature_extractor": [
      "transformers",
      "CLIPImageProcessor"
    ],
    "image_encoder": [
      null,
      null
    ],
    "requires_safety_checker": true,
    "safety_checker": [
      "stable_diffusion",
    